conda env list
conda create -n ml_final python="3.10"
conda activate ml_final
pip install ipykernel
python -m ipykernel install --user --name ml_final //주의: 디스플레이는 생략해도 작동합니다
pip install pandas numpy scikit-learn //주의: sklearn이라고 쓰면 안돼요

In [1]:
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score 
#민스퀘어 에러가 평균 제곱 오차 구하는 함수입니다 r2스코어는 결정계수를 구하고 accuracy_score는 문제의 정답률을 구하는 함수입니다
import pandas #판다스의 함수중에 중요한 함수는 read_csv와 DataFrame과 dropna입니다
import numpy #넘파이에 중요한 함수는 hstack입니다 데이터를 연결시키는 조인같은 함수입니다
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [2]:
data_url = "https://lib.stat.cmu.edu/datasets/boston"
raw_data = pandas.read_csv(data_url,sep='\s+',skiprows=22,header=None)
data = numpy.hstack([raw_data.values[::2,:],raw_data.values[1::2,:2]])
#Boston 주택 가격 데이터(raw_df.values)는 홀수 줄과 짝수 줄로 쪼개져 있는 특징이 있어서 hstack으로 데이터를 이어 주어야 합니다 
#그래서 홀수줄과 짝수줄을 분리하고 데이터를 이은 겁니다
target = raw_data.values[1::2,2] # 정답 데이터를 따로 뺴놓은 것입니다

df = pandas.DataFrame(data)
df['MEDV'] = target #데이터프레임(df)에 'MEDV'라는 이름의 새로운 열(Column)을 만들고, 거기에 정답 데이터(target)를 집어넣어라 라는 코드입니다
clining_data = df.dropna(axis=0) 
#dropna(결측치 제거)는 판다스 형식의 데이터에만 작동되기에 넘파이 함수 hstack으로 변환된 데이터를 한번더 DataFrame으로 판다스 형식으로 바꾼후 실행해야 합니다

x = df.iloc[:,:13] #데이터프레임(df)에서 맨 앞(0번)부터 12번 열까지, 총 13개의 열(문제지)만 쏙 뽑아서 x에 저장하라는 코드입니다
y = df['MEDV'] #그럼 이건 정답을 y에 넣으라는 소리겠죠??

In [3]:
mid_values = y.median()#전체 주택 가격 데이터(y)의 중앙값(가운데 위치한 값)을 찾아서 mid_values에 저장해라 라는 말입니다
y_class = (y >= mid_values).astype(int) #이거는 y(정답값)이 전체 주택 가격 데이터(y)의 중앙값(가운데 위치한 값)보다 크거나 같으면 True 아니면False를 반환하는데 
#이거의 타입을 정수로 저장하라는것입니다 참고로 불리언을 변환하면 트루는 1이고 아니라면 ㅎㅎ 아시겠죠?? 

x_train_reg,x_test_reg,y_train_reg,y_test_reg = train_test_split(x,y,test_size=0.2,random_state=42)
x_train_cls,x_test_cls,y_train_cls,y_test_cls = train_test_split(x,y_class,test_size=0.2,random_state=42)
#train_test_split는 데이터를 학습용과 검증용으로 쪼개줍니다 test_size를 0.2로 잡으면 자연스럽게 학습용데이터는 0.8로 80% 20%나뉩니다
#random_state=42는 랜덤값을 고정해주는 함수로 
#42는 딱히 별의미는 없고 아마 은하수를 여행하는 히치하이커를 위한 안내서라는 소설에서 뜬금없이 어려운 문제의 답이 42가 나와서 밈이 되었는데 
#그게 이런 랜덤값에 이스터에그로 많이 쓰인다고 하네요 (믿거나 말거나)

In [8]:
scale_reg = StandardScaler() #여기서 StandardScaler()는 데이터를 0~1사이로 조절하는 기능입니당

x_train_reg_scale = scale_reg.fit_transform(x_train_reg) # 그러면 이건 x_train_reg_scale에 x_train_reg를 StandardScaler로 조절한 값을 넣는 코드겠죠
x_test_reg_scale = scale_reg.transform(x_test_reg) #여기도 뭐 위에랑 비슷한거고
# 아 그리고 fit_transform()는 데이터의 평균과 표준편차를 학습(fit)함과 동시에 데이터를 변환(transform)해주는 함수입니다 
#이걸 하면 _이 붙은 변수가 조용히 생성된다고 하는데 우리가 직접 쓸일은 없을거에요
#그러나 여기서 중요한점 절대 절대 x_test_reg는 fit를 하면 안됩니다
scale_cls = StandardScaler() 

x_train_cls_scale = scale_cls.fit_transform(x_train_cls)
x_test_cls_scale = scale_cls.transform(x_test_cls)

In [22]:
lr_model = LinearRegression() # LinearRegression()는 회귀 모델을 만들어줍니다
lr_model.fit(x_train_reg_scale,y_train_reg) #fit(x_train, y_train)은 모델에게 문제(x_train)와 정답(y_train)을 주고 학습 시키는 함수입니다

y_pridic = lr_model.predict(x_test_reg_scale) #predict(X_test)은 이제 학습을 다 하면 새로운 문제를 주고 그 값을 예측시키는 함수입니다

lr_r2_score = r2_score(y_test_reg,y_pridic) #r2_score는 정답(y_test_reg) 과 예측값(y_pridic)을 넣어서 결정계수를 확인하는 검증 함수입니다
lr_mse = mean_squared_error(y_test_reg,y_pridic) #mean_squared_error는 정답(y_test_reg) 과 예측값(y_pridic)을 넣어서 평균제곱오차를 확인하는 검증 함수입니다

print(lr_r2_score)#이건뭐 말 안해도 알거고 ㅎㅎ
print(lr_mse)

0.668759493535632
24.291119474973513


### [선형회귀 결과 해석]
선형 회귀 모델 평가 결과 테스트 데이터에 대한 결정계수는 약 0.6687로 모델이 전체 데이터 변동성의 약 66.9%를 설명하고 있음을 보여줍니다 평균제곱오차는 약 24.29로 나타나며 전반적으로 주택 가격의 경향성을 유의미하게 예측하고 있습니다

In [21]:
lo_model = LogisticRegression(max_iter=10000) # LogisticRegression()는 회귀 모델을 만들어줍니다 여기서 max_iter는 최대 학습 횟수입니다
lo_model.fit(x_train_cls_scale,y_train_cls)#알겠죠?? 이거 위에서 0과 1로 데이터 변환한거를 학습시키는거 알고 계시죠??
y_pridic = lo_model.predict(x_test_cls_scale) # 이건뭐 아실거고 위에서 설명했으니까

lo_acc = accuracy_score(y_test_cls,y_pridic) #이거이거 중요합니다 분류 모델이 0과 1을 얼마나 정확하게 맞췄는지 정확도를 구해주는 정답률 확인 함수입니다

print(lo_acc)#이건도 뭐 말 안해도 알거고 ㅎㅎ

0.8725490196078431


In [24]:
nn_model = MLPRegressor(hidden_layer_sizes=(64,32),max_iter=10000,random_state=42)
#이거 이제 아시겠죠?? 히든레이어사이즈는 말 그대로 은닉층 어떻게 만들지 물어보는거고 다른거는 설명했으니까...
nn_model.fit(x_train_reg_scale,y_train_reg)# 이것도 뭐 학습시키는거...

y_pridic = nn_model.predict(x_test_reg_scale) #이것도 뭐 실제로 예측해보라고 시키는거...

nn_r2_score = r2_score(y_test_reg,y_pridic)#이건 뭐 결정계수...
nn_mse = mean_squared_error(y_test_reg,y_pridic)# 그럼 이건 뭐 평균제곱 오차겠네요...
print(nn_r2_score)# 말안해도 다아실거고 ㅎㅎ
print(nn_mse)

0.8311070843365629
12.385556454577207


### [선형 회귀와 인공신경망 결과 비교 분석 의견]
우선 인공신경망의 평균오차가 선형 회귀의 평균오차보다 적습니다 이는 인공신경망의 예측이 더 우수함을 보여줍니다
주택 가격 데이터셋 내부의 특성치들과 타겟 간의 관계는 복잡한 비선형적 관계를 맺고 있습니다 선형 회귀는 이를 직선적인 관계로만 제한하여 학습하지만 인공신경망은 은닉층과 활성화 함수를 통해 복잡한 비선형 패턴을 유연하게 포착해 낼 수 있기 때문입니다